In [7]:
import os

print(os.listdir("/content"))

['.config', 'customer-churn-training.csv', 'sample_data']


In [8]:
import pandas as pd

df = pd.read_csv("/content/customer-churn-training.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Dataset loaded successfully!
Dataset shape: (12, 7)
Columns: ['customer_id', 'tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type', 'churned']


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


In [9]:
X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]

print("Input features:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

Input features:
['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type']

Target distribution:
churned
0    7
1    5
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 9
Testing rows: 3


In [11]:

# Separate numerical and categorical columns
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

# Preprocessing for numerical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing for categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Preprocessing pipeline created successfully!")


Preprocessing pipeline created successfully!


In [12]:

# Create the machine-learning model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train the model using training data
model.fit(X_train, y_train)

print("Model trained successfully!")


Model trained successfully!


In [13]:

# Predict customer churn for test data
y_pred = model.predict(X_test)

# Display model performance
print("Model Evaluation:")
print(classification_report(y_test, y_pred, zero_division=0))


Model Evaluation:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



In [14]:

# Get feature names after preprocessing
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

# Get feature importance scores
importances = model.named_steps["classifier"].feature_importances_

# Create a table of feature importance
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

display(importance_df)

print("Feature importance analysis completed!")


,Feature,Importance
1,numeric__support_tickets,0.269750
0,numeric__tenure_months,0.243750
2,numeric__monthly_spend_inr,0.170000
4,categorical__plan_type_Basic,0.163750
3,numeric__last_login_days,0.116429
5,categorical__plan_type_Pro,0.030071
6,categorical__plan_type_Standard,0.006250


Feature importance analysis completed!
